## Import

In [13]:
import os
import time
import numpy as np
import scipy.linalg as linalg
import torch
import torch.nn as nn
import torchvision as tv
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torch.nn.functional as F
import torch.optim as optim
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
#import preprocess


device = 'cuda:1'

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Dataset

In [4]:
from datasets.Student_t import MultivariateStudentT

d = 64
dim_x = dim_y = d//2
rho = 0.7
df = 1
mean = np.zeros(dim_x + dim_y)
V = np.eye(dim_x)*rho
dispersion = np.eye(dim_x + dim_y)
dispersion[0:dim_x, :][:, dim_x:] = V
dispersion[dim_x:, :][:, 0:dim_x] = V


dataset = MultivariateStudentT(
        dim_x=dim_x,
        dim_y=dim_y,
        mean=mean,
        dispersion=dispersion,
        df=df)

X, Y = dataset.sample(10000)
X, Y = torch.Tensor(X).float().to(device), torch.Tensor(Y).float().to(device)

print(f'MI is {dataset.mutual_information(): 0.2f}')
print('X', X.shape, 'Y', Y.shape)

MI is  12.46
X torch.Size([10000, 32]) Y torch.Size([10000, 32])


## Hyperaparams

In [6]:
class Hyperparams(object):
    def __init__(self): 
        self.critic = 'neural'                # ('neural', 'quadratic')
        self.lr = 5e-4
        self.bs = 500
        self.n_bridges = 4
        self.wd = 1e-5
        
hyperparams=Hyperparams()

architecture_critic = [d, 500, 500, 500, 1]
architecture_encode = [d//2, 200, 200, d//2]

## Separate learning

In [14]:
## Vector copula estimation
from estimators.VCE import VCE

hyperparams.K_components = 5
hyperparams.nde_type = 'VGC'

ret = []
time_spend = []
for i in range(8):
    t0 = time.time()
    
    estimator = VCE(None, None, architecture_critic, hyperparams)
    estimator.to(device)
    estimator.learn(X, Y)

    mi = estimator.MI(X, Y)
    t = time.time()-t0

    print('true MI:', dataset.mutual_information())
    print('est MI:', estimator.MI(X, Y))
    
    ret.append(mi)
    time_spend.append(t)

K components= 5 copula transform= True
nde type: VGC
finished: t= 0 loss= 202.3946990966797 loss val= 211.05810546875 best val loss= 211.05810546875 best t= 0
finished: t= 51 loss= -52.313987731933594 loss val= 70626.8515625 best val loss= -0.14143085479736328 best t= 2
finished: t= 102 loss= -56.493690490722656 loss val= 15092.9716796875 best val loss= -0.14143085479736328 best t= 2


finished: t= 0 loss= 77.45835876464844 loss val= 81.28773498535156 best val loss= 81.28773498535156 best t= 0
finished: t= 51 loss= -50.71923828125 loss val= -51.04662322998047 best val loss= -51.04662322998047 best t= 51
finished: t= 102 loss= -53.420677185058594 loss val= -51.077850341796875 best val loss= -51.79608154296875 best t= 91
finished: t= 153 loss= 3615.48779296875 loss val= 7266.77294921875 best val loss= -51.963897705078125 best t= 112
finished: t= 204 loss= 3653.045654296875 loss val= 3436.0517578125 best val loss= -51.963897705078125 best t= 112
finished: t= 255 loss= 11.854382514953613 l

In [20]:
ret = sorted(ret)

print(np.array(ret)[len(ret)//2], np.array(ret).std(), np.array(ret).max(), np.array(ret).min())
print(np.array(time_spend).mean(), np.array(time_spend).std())

9.50607967376709 0.3515723632296572 10.028558731079102 8.793410301208496
265.8355876505375 30.261200230704247
